## `░ 1. Instalación de librerias`

In [ ]:
# %pip install sqlalchemy
# %pip install pandas
# %pip install oracledb 
%pip install openpyxl

#### `» 1.1 Importar librerias`

In [1]:
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.pool import NullPool
import oracledb

## `░ 2. Metodos globales y constantes`

#### `» 2.1 Constantes`

In [1]:
# Conexión:

USER = 'USRSEGURIDAD'
PASSWORD = 'C0ntr4s3n4$3gur4'
HOST = '172.19.0.42'
PORT = '1521'
SID = 'INABIF02'

#### `» 2.1 Conexión a la base de datos Oracle`

In [2]:
def get_engine():
   dsn = oracledb.makedsn(HOST, PORT, sid=SID)
   connection_string = f"oracle+oracledb://{USER}:{PASSWORD}@{dsn}"
   return create_engine(
      connection_string,
      poolclass=NullPool,  # ← AGREGADO: Sin pool para evitar problemas
      echo=False)  # Cambiar a True para debug)

#### `» 2.2 Métodos globales`

In [4]:
# 2.2.1 Insertar DataFrame por lotes a la base de datos Oracle
def  insertar_dataframe_por_lotes(df, table_name, lote_size=1000):
   try:
         engine = get_engine()
         with engine.begin() as conn: # Maneja automáticamente commit/rollback
            df.to_sql(name=table_name, con=conn, if_exists='append', index=False, chunksize=lote_size)
            
         print(f'Datos insertados exitosamente en la tabla {table_name}.')
         return True, len(df)
   except Exception as e:
         print(f'Error al insertar datos: {e}')
         return False, 0
   finally:
         engine.dispose()
      

# 2.2.2 Ejecutar consulta y retornar DataFrame
def  execute_query(query, params=None):
   try:
         engine = get_engine()
         with engine.connect() as conn: # Cierra automáticamente la conexión
            if params:
               df = pd.read_sql_query(sql=text(query), con=conn, params=params)
            else:
               df = pd.read_sql_query(sql=text(query), con=conn)
            
         print(f'Consulta ejecutada exitosamente, filas encontradas: {len(df)}')
         return df
   except Exception as e:
         print(f"Error al ejecutar la consulta: {e}")
         return df.DataFrame()

## `░ 3. Exportar datos de oracle DB a archivos CSV`

#### `» 3.1 Constantes:`

In [5]:
PATH_BASE = './data'

#### `» 3.2 Consulta sql para extraer datos de de Oracle DB`

In [6]:
# Test
QUERY = '''
            DECLARE
            c_resultado SYS_REFCURSOR;
         BEGIN
            USP_GENERAR_REPORTE_MATRIZ_FAMILIAS_IGUALITARIAS(c_resultado);
            DBMS_SQL.RETURN_RESULT(c_resultado);
         END;

'''

#df_result = execute_query(QUERY, params={'anexo_id': 1050})
df_result = execute_query(QUERY)

Error al ejecutar la consulta: name 'oracledb' is not defined


UnboundLocalError: cannot access local variable 'df' where it is not associated with a value

In [9]:
df_result.to_excel(f'{PATH_BASE}/TGUNIDADORGANICA.xlsx', index=False)